In [1]:
import pysam
from collections import defaultdict
import pandas as pd

In [2]:
whitelist_df = pd.read_csv("../../top-100_10X-capture/top100_whitelist.tsv", sep="\t")

location_whitelist = (whitelist_df["location_chr"].astype(str) + "-" +
                      whitelist_df["location_start"].astype(str) + "-" +
                      whitelist_df["barcode"].astype(str)).tolist()

In [3]:
def t7_associations(bam_path):
    bam = pysam.AlignmentFile(bam_path, "rb")

    # defining the upstream sequence that
    # should be used for pattern matching in
    # order to extract the barcode sequence
    motif = "CTTTCCGCGT"

    motif_len = len(motif)
    barcode_len = 7

    # counters
    read_counts = defaultdict(int)
    umi_sets = defaultdict(set)

    # iterating through the cellranger bam file
    for read in bam.fetch(until_eof=True):

        # skipping secondary and supplementary reads
        if read.is_secondary or read.is_supplementary:
            continue

        # getting the tags from the bam file
        tags = dict(read.tags)

        # getting the cellranger-corrected 
        # cell barcode and UMI sequences
        cb = tags.get("CB", None)
        ub = tags.get("UB", None)

        # skipping reads that don't have a cell barcode
        # or a UMI
        if cb is None or ub is None:
            continue

        # getting the actual sequencing read
        seq = read.query_sequence

        # finding the position in the sequence
        # that contains the pattern before the 
        # barocde sequence
        pos = seq.find(motif)

        # only process reads where the sequence
        # contains the pattern
        if pos != -1:
            # positionally extracting the barcode sequence
            # using the pattern match
            edit_bc = seq[pos+motif_len:pos+motif_len+barcode_len]
        else:
            continue

        # getting the genomic start location of the aligned reads
        chrom = read.reference_name 
        start = read.reference_start+1

        # skipping reads without mapped chromsome or start location
        if chrom is None or start is None:
            continue

        # defining a "location_barcode" as the genomic read start
        # plus the edit barcode
        location_barcode = chrom + "-" + str(start) + "-" + edit_bc

        # checking if the location_barcode is in a whitelist
        # of valid location_barcodes from bulk T7 mapping data
        if location_barcode in location_whitelist:        
            # if it is, creating a dictionary
            # key that is the cell barcode plus
            # the location_barcode
            key = (cb, location_barcode)

            # incrementing the read count and umi count
            read_counts[key] += 1
            umi_sets[key].add(ub)  

    bam.close()

    # build dataframe
    rows = []
    for (cb, location_barcode), n_reads in read_counts.items():
        n_umis = len(umi_sets[(cb, location_barcode)])
        rows.append((cb, location_barcode, n_reads, n_umis))

    count_df = pd.DataFrame(rows, columns=["cbc", "location_barcode", "reads", "umis"])
    
    return count_df 

In [4]:
bam_path = "./count_t7/t7_K562_L1/outs/possorted_genome_bam.bam"
k562_l1 = t7_associations(bam_path)
k562_l1.to_csv("t7_K562_L1.associations.tsv", sep="\t", index=False)

bam_path = "./count_t7/t7_K562_L2/outs/possorted_genome_bam.bam"
k562_l2 = t7_associations(bam_path)
k562_l2.to_csv("t7_K562_L2.associations.tsv", sep="\t", index=False)